# 1D CNN-A — Data Pipeline & Autoencoder Training

Loads TSLA 1-minute bars from CSV, engineers 14 technical-indicator features,
scales with `RobustScaler`, slices into overlapping 64-bar windows, filters
out gap-spanning windows, then trains a 1D CNN autoencoder with a 6-condition
`TrainingGuard`. Saves `model.pt` to `DATA_DIR / SYMBOL /` on completion.

> **Run this notebook first.** `latent_cluster.ipynb`, `reconstruction.ipynb`,
> `cluster_quality.ipynb`, and `temporal_patterns.ipynb` all depend on `model.pt`.

## 1. Imports

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # macOS: prevents libiomp/libomp conflict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from config import Config
from data import (
    load_bars, clean_data, add_features, drop_feature_nans,
    scale_features, make_windows, filter_gap_windows,
)
from model import ConvAutoencoder, make_dataloaders, save_model
from guard import TrainingGuard

print(f"PyTorch {torch.__version__} | device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Config

In [ ]:
from config import Config

cfg = Config()

# ── Override defaults here before running the rest of the notebook ────────────
# cfg.MAX_BARS       = None   # load all bars (~552k)
# cfg.EPOCHS         = 30     # full training run
# cfg.N_CLUSTERS     = 12     # try more/fewer clusters
# cfg.LATENT_DIM     = 64     # larger latent space
# cfg.N_SAMPLE       = 5_000  # render more windows in Section 9

# Expose all config fields as module-level names so every downstream cell
# can use SYMBOL, WINDOW_SIZE, LR, feature_cols, DEVICE, etc. unchanged.
globals().update(vars(cfg))

print(f"Symbol={SYMBOL}  Timeframe={TIMEFRAME}  {START_DATE} → {END_DATE}")
print("Using device:", DEVICE)

## 3. Fetch Data from Alpaca API
Calls the local `alpaca_api` FastAPI service (must be running: `uv run main.py`).
Fetches TSLA 1-minute bars and saves to both DB and CSV.

In [ ]:
if FETCH_DATA:
    import httpx  # only needed when FETCH_DATA = True
    params = {
        "symbols": SYMBOL,
        "timeframe": TIMEFRAME,
        "start": START_DATE,
        "end": END_DATE,
        "save_to": "db,csv",
    }
    with httpx.Client(timeout=None) as client:
        r = client.get(f"{API_BASE}/bars", params=params)
        r.raise_for_status()
        result = r.json()
    bars = result.get("data", {}).get("bars", {}).get(SYMBOL, [])
    print(f"Fetched {len(bars)} bars for {SYMBOL}")
    print("Saved:", result.get("saved"))
else:
    print("FETCH_DATA=False — skipping. Set True in Config to re-pull.")

## 4. Load Data

In [ ]:
# Load raw OHLCV bars from the CSV file.
df = load_bars(DATA_DIR, SYMBOL, TIMEFRAME, MAX_BARS)

Check for:

Duplicate timestamps
Missing timestamps (gaps)
NaNs
Infinite values
Bad OHLC relationships (high < low, etc.)

Typical checks:

In [ ]:
# Remove duplicate timestamps and rows with missing values.
df = clean_data(df)

## 5. Verify Time Continuity
A CNN assumes a consistent sequence.

Look for:

missing bars
duplicate bars
irregular spacing

If you're using 1-minute candles, every row should be exactly 1 minutes apart.

In [ ]:
delta = df["timestamp"].diff()
dt = pd.to_timedelta(delta).dt.total_seconds()
print("Average time delta (seconds):", dt.mean())
print("Time delta distribution (seconds):")
print(dt.describe())

## 6. Add Features

In [ ]:
# Calculate 14 technical indicator columns (EMAs, MACD, candle shape, returns, volume ratio).
df = add_features(df)

### 6b. Remove Warm-Up NaN Rows

Feature engineering creates NaNs.

In [ ]:
# Drop the warm-up rows where EMAs and rolling means don't have enough history yet.
df = drop_feature_nans(df)

## 7. Scale Features

This is critical.

CNNs train poorly on:

close = 45000
volume = 10000000
return = 0.001

all mixed together.

StandardScaler

Most common:

In [ ]:
# Normalise every feature column so they all sit in a similar numeric range.
# RobustScaler uses the median and IQR — better than mean/std for financial data with outliers.
df, scaler = scale_features(df, feature_cols)

## 8. Create Fixed-Length Windows

A CNN does not ingest an entire dataframe.

It ingests samples.

In [ ]:
# Slice the time series into overlapping WINDOW_SIZE-bar windows.
# Each window is one training sample for the CNN.
X_raw = make_windows(df, feature_cols, WINDOW_SIZE)
n_features = len(feature_cols)

## 9. Filter Gap Windows
A window that spans an overnight or weekend gap mixes pre-gap and post-gap bars — the CNN would learn noise, not patterns. Any window whose 64-bar span crosses a gap > 5 minutes is dropped.

In [ ]:
# Remove windows that span overnight or weekend gaps.
# Such windows would teach the model noise rather than real patterns.
X_clean, valid_mask = filter_gap_windows(X_raw, df, WINDOW_SIZE)

If You Also Have a Target Column

Suppose:

In [ ]:
# feature_cols = [
#     "open", "high", "low", "close", "volume",
#     "ema20", "ema50", "rsi14", "atr14", "adx14"
# ]

# target_col = "target"

# WINDOW_SIZE = 64

# features = df[feature_cols].to_numpy(dtype=np.float32)
# targets = df[target_col].to_numpy()

# X = np.array([
#     features[i:i + WINDOW_SIZE]
#     for i in range(len(features) - WINDOW_SIZE)
# ])

# y = targets[WINDOW_SIZE:]

# print(X.shape)
# print(y.shape)

# (936, 64, 10)
# (936,)

# X[0] = bars 0-63
# y[0] = target at bar 64

# X[1] = bars 1-64
# y[1] = target at bar 65

## 10. Train/Test Split & DataLoader

In [ ]:
# Split windows chronologically — first 80% for training, last 20% for validation.
# Wrap each split in a DataLoader so PyTorch can iterate them in batches.
train_loader, test_loader = make_dataloaders(X_clean, TEST_SPLIT, BATCH_SIZE)

## 11. Autoencoder Model
Encoder compresses `(batch, 14, 64)` → latent vector `(batch, LATENT_DIM)`.
Decoder reconstructs `(batch, 14, 64)` from the latent vector.
Training loss is reconstruction MSE — no labels needed.

In [ ]:
# ConvAutoencoder is defined in model.py — import it at the top.
# Here we just create an instance and move it to the device (CPU or GPU).
model = ConvAutoencoder(n_features=n_features, latent_dim=LATENT_DIM).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")
print(model)

## 12. Training Guard — When to Stop Early

### What is Learning Rate?

Learning rate (`LR`) controls **how big a step** the optimizer takes each time it updates the model's weights after processing a batch:

- **Too high** — the update overshoots the loss minimum. The model jumps past the best weights and lands somewhere worse each step. You'll see loss explode or bounce up and down.
- **Too low** — updates are tiny. Training crawls, stalls early, or never reaches a good minimum within your epoch budget.
- **Just right** — loss falls smoothly and steadily each epoch, then levels off when the model converges.

A rule of thumb for the **Adam optimizer**: start at `1e-3`. If training is unstable, try `1e-4`. If it's converging too slowly, try `3e-3`. Those three values cover the realistic range for most autoencoders — `LR = 1e-3` is Adam's own default for a reason.

---

Six things can go wrong silently during training — some waste hours of time, others produce a model that learned nothing at all. The cells below show you what each failure looks like, then give you a `TrainingGuard` class that catches all of them automatically during the training loop.

| # | Condition | What it looks like | Why it happens | Fix |
|---|-----------|-------------------|----------------|-----|
| 1 | **NaN / Inf loss** | Loss suddenly becomes `nan` or `inf` | A bad batch drove weights to ±infinity | Lower `LR`; add gradient clipping |
| 2 | **Loss explosion** | Loss *grows* each epoch instead of shrinking | `LR` too high — model overshoots the minimum every step | Reduce `LR` by 10× and restart |
| 3 | **Reconstruction collapse** | Loss hits near-zero within 2–3 epochs | Decoder outputs the mean of all inputs — trivially low MSE, zero insight | Inspect reconstructions; they'll all look identical |
| 4 | **Overfitting** | Train loss falls, val loss rises | Model memorised specific 64-bar windows, not general patterns | Add `Dropout(0.2)`, reduce `LATENT_DIM` |
| 5 | **Plateau** | Val loss flat for N epochs | Model has converged — extra epochs waste time | Stop early; best weights already found |
| 6 | **Oscillation** | Loss bounces up and down each epoch | `LR` too high — jumping back and forth over the minimum | Reduce `LR` by 2–5× |

In [ ]:
# ── Diagnostic Illustrations ────────────────────────────────────────────────
np.random.seed(42)
e = np.arange(1, 31)

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()

# 1. Healthy
ax = axes[0]
tr = 0.5 * np.exp(-0.12 * e) + 0.02
vl = 0.55 * np.exp(-0.10 * e) + 0.03
ax.plot(e, tr, 'royalblue', lw=2, label='train loss')
ax.plot(e, vl, 'tomato',    lw=2, label='val loss')
ax.set_title('[OK]  Healthy Training', fontsize=11, color='green', fontweight='bold')
ax.set_ylim(0, 0.65); ax.legend(fontsize=8)

# 2. Loss Explosion
ax = axes[1]
tr_ok = 0.5 * np.exp(-0.12 * e[:12])
boom  = [tr_ok[-1] * f for f in [1, 2, 6, 18, 55]]
ax.plot(e[:12],   tr_ok, 'royalblue', lw=2,   label='train loss')
ax.plot(e[11:16], boom,  'red',       lw=2.5, label='explosion')
ax.axvline(12, color='red', ls='--', alpha=0.5)
ax.set_title('[STOP]  Loss Explosion / NaN', fontsize=11, color='red', fontweight='bold')
ax.set_ylim(0, 60); ax.legend(fontsize=8)
ax.annotate('LR too high\nor bad data', xy=(13, 32), fontsize=9, color='red')

# 3. Overfitting
ax = axes[2]
tr      = 0.5 * np.exp(-0.15 * e) + 0.01
base_vl = 0.52 * np.exp(-0.12 * 12)
vl      = np.concatenate([0.52 * np.exp(-0.12 * e[:12]),
                           base_vl + 0.012 * np.arange(1, 19)])
ax.plot(e, tr, 'royalblue', lw=2, label='train loss')
ax.plot(e, vl, 'tomato',    lw=2, label='val loss')
ax.axvline(12, color='orange', ls='--', alpha=0.7, label='diverges here')
ax.fill_between(e[11:], tr[11:], vl[11:], alpha=0.15, color='orange')
ax.set_title('[WARN]  Overfitting', fontsize=11, color='darkorange', fontweight='bold')
ax.legend(fontsize=8)

# 4. Plateau
ax = axes[3]
tr = np.concatenate([0.5 * np.exp(-0.25 * e[:8]),
                     np.full(22, 0.11) + 0.002 * np.random.randn(22)])
vl = tr + 0.025
ax.plot(e, tr, 'royalblue', lw=2, label='train loss')
ax.plot(e, vl, 'tomato',    lw=2, label='val loss')
ax.axvline(8, color='purple', ls='--', alpha=0.7)
ax.annotate('plateaued\nhere', xy=(9, 0.19), fontsize=9, color='purple')
ax.set_title('[STOP]  Plateau — No Improvement', fontsize=11, color='purple', fontweight='bold')
ax.legend(fontsize=8)

# 5. Oscillation
ax = axes[4]
base  = 0.5 * np.exp(-0.04 * e) + 0.1
noisy = base + 0.08 * np.sin(e * 2.5) + 0.04 * np.random.randn(30)
ax.plot(e, noisy, 'royalblue', lw=1.8, label='train (actual)', alpha=0.9)
ax.plot(e, base,  'g--',       lw=1.5, label='expected trend', alpha=0.7)
ax.set_title('[WARN]  Oscillation — LR Too High', fontsize=11, color='darkorange', fontweight='bold')
ax.legend(fontsize=8)

# 6. Reconstruction Collapse
ax = axes[5]
collapse = np.concatenate([[0.48, 0.05, 0.002], np.full(27, 8e-5)])
ax.plot(e, collapse, 'royalblue', lw=2, label='train loss')
ax.axhline(1e-4, color='red', ls='--', alpha=0.7, label='collapse threshold')
ax.set_yscale('log')
ax.set_title('[STOP]  Reconstruction Collapse', fontsize=11, color='darkred', fontweight='bold')
ax.legend(fontsize=8)
ax.annotate('output = average\nof all windows', xy=(15, 2e-4), fontsize=9, color='darkred')

for ax in axes:
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.grid(alpha=0.25)

plt.suptitle('Training Failure Patterns — What TrainingGuard Watches For',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


# TrainingGuard is defined in guard.py and imported at the top.
# Instantiate it here using the values from config.py (GUARD_* fields).
guard = TrainingGuard(
    patience=GUARD_PATIENCE,
    min_delta=GUARD_MIN_DELTA,
    overfit_ratio=GUARD_OVERFIT_RATIO,
    explosion_factor=GUARD_EXPLOSION_FACTOR,
    oscillation_window=GUARD_OSCILLATION_WINDOW,
    oscillation_cv=GUARD_OSCILLATION_CV,
    collapse_threshold=GUARD_COLLAPSE_THRESHOLD,
)
print("TrainingGuard ready.")
print(f"  patience={guard.patience}  |  min_delta={guard.min_delta}  |  "
      f"overfit_ratio={guard.overfit_ratio}x  |  explosion={guard.explosion_factor}x")

## 13. Train (Reconstruction Loss)

### The Optimizer and Learning Rate

```python
optimizer = torch.optim.Adam(model.parameters(), lr=LR)  # LR = 1e-3
```

**Adam** (Adaptive Moment Estimation) is the standard first choice for autoencoders. Unlike plain SGD — which applies the same step size to every weight — Adam adapts the step size *per weight* based on how consistently that weight has been moving. Weights that haven't changed much get larger steps; weights that are already oscillating get smaller ones. `LR` sets the baseline scale for all those individual adjustments.

---

### Do you need a chart for learning rate?

With a **fixed `LR = 1e-3`** (what we're using here) — **no**. It's a single constant for the entire run; there's nothing to plot.

A chart becomes useful only when you add a **learning rate scheduler** — code that adjusts `LR` automatically during training in response to what's happening:

| Scheduler | What it does | When to use |
|-----------|-------------|-------------|
| `ReduceLROnPlateau` | Halves `LR` when val loss stops improving for N epochs | Best default — automatic and safe |
| `CosineAnnealingLR` | Smoothly decays `LR` from start value down to near zero | Longer runs with a fixed epoch budget |
| `OneCycleLR` | Warms `LR` up then decays it (super-convergence) | When you need faster convergence in fewer epochs |

If you add a scheduler, you'd log `optimizer.param_groups[0]['lr']` each epoch and plot it on a second y-axis alongside `train_loss` and `val_loss`. For now, `LR = 1e-3` is fixed — the **Reconstruction Loss chart below tells you everything you need**.

In [ ]:
import os
from tqdm import tqdm

torch.set_num_threads(os.cpu_count())
print(f"PyTorch using {torch.get_num_threads()} threads")

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
guard.stop_reason = None   # reset so cell is safe to re-run

train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(batch), batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))

    model.eval()
    with torch.no_grad():
        val_loss = sum(criterion(model(b.to(DEVICE)), b.to(DEVICE)).item()
                       for b in test_loader) / len(test_loader)
    val_losses.append(val_loss)

    print(guard.status(epoch, train_losses[-1], val_loss))

    if guard.check(epoch, train_losses[-1], val_loss):
        print(f"\n{guard.stop_reason}")
        break
else:
    print(f"\nCompleted all {EPOCHS} epochs.")

plt.figure(figsize=(10, 3))
plt.plot(train_losses, label="train")
plt.plot(val_losses,   label="val")
plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.legend(); plt.title("Reconstruction Loss")
plt.tight_layout(); plt.show()

# ── Save checkpoint ────────────────────────────────────────────────────────
# save_model() is defined in model.py — it handles directory creation and saving.
save_model(model, DATA_DIR, SYMBOL)
print(f"  n_features={n_features}  WINDOW_SIZE={WINDOW_SIZE}  "
      f"LATENT_DIM={LATENT_DIM}  LR={LR}")

### Checkpoint saved

**File:** `DATA_DIR / SYMBOL / model.pt`  — e.g. `../../data/TSLA/model.pt`  
**Format:** `model.state_dict()` — weight tensors only, not a pickled model object

| What | Detail |
|------|--------|
| Class | `ConvAutoencoder(n_features, LATENT_DIM)` |
| Input tensor | `(batch, n_features, WINDOW_SIZE)` — channels-first |
| `n_features` | `len(feature_cols)` — 14 by default |
| `WINDOW_SIZE` | bars per window — 64 by default |
| `LATENT_DIM` | bottleneck size — 32 by default |
| Optimizer | Adam, `lr=LR` |
| Loss | `nn.MSELoss()` — mean squared reconstruction error |
| Feature scaling | `RobustScaler` per feature column (Section 7) |

**To reload in `latent_cluster.ipynb`:**
```python
model = ConvAutoencoder(n_features=n_features, latent_dim=LATENT_DIM).to(DEVICE)
model.load_state_dict(torch.load(model_path, map_location=DEVICE))
model.eval()
```

**Next step:** open `latent_cluster.ipynb` — it loads this checkpoint and runs
Sections 15 (Extract Latent Vectors) and 16 (Cluster & Visualise Patterns).